This lesson is about CNNs, how they're different from classic NNs, when to use  them, and an example of digit classification. Also in this notebook are concepts such as Convolution, Pooling

In [1]:
import torch
import torchvision
from torch.utils.data import DataLoader
from torch import nn, optim

In [2]:
# Use GPU instead of CPU for faster results
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
torch.manual_seed(42)

In [4]:
# MNIST contains 28x28 pixel images.
# In this example, they are greyscale images where each pixel is represented
# by a number from 0 to 1.
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=torchvision.transforms.ToTensor()
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=torchvision.transforms.ToTensor()
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 459kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.46MB/s]


In [5]:
# Load the train/test data into DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False
)

In [6]:
# Print out the shape of one image
for image, label in train_loader:
  print(f'Shape: {image.shape}, Label: {label}')
  break

Shape: torch.Size([64, 1, 28, 28]), Label: tensor([1, 2, 8, 5, 2, 6, 9, 9, 9, 4, 0, 3, 9, 9, 5, 6, 7, 8, 8, 9, 2, 6, 9, 3,
        0, 5, 0, 7, 6, 1, 2, 0, 7, 4, 6, 0, 6, 9, 7, 0, 7, 3, 2, 5, 9, 0, 4, 8,
        3, 6, 4, 0, 3, 2, 6, 6, 3, 2, 2, 3, 6, 7, 8, 4])


Shaoe of the batch is [64, 1, 28, 28] where 64 is the number of imgs per batch.
1 is the amount of numbers we need to represent color (1 with greyscale, if it was RGB it would have been 3), 28x28 is the image dimension

In [7]:
# Create the CNN model
cnn_model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), # 1  - num of input channels (greyscale=1 (1 value), rgb=3 (red, green, blue) -> 3 values),
                                                # 16 - num of output channels (matrices), each one detects a different pattern (edge, diagonal line, curve etc.)
                                                # 3 - kernel_size -> size of each filter matrix (3x3 in this case)
                                                # 1 - padding adds 1 layer of 0s around the image so that the resulting conv image is 28x28 and not 26x26
    nn.ReLU(),

    # Select the 'brightest' pixel from every 2x2 square in our image
    nn.MaxPool2d(2), # After this image shape is 14x14

    nn.Conv2d(16, 32, kernel_size=3, padding=1), # image is 14x14, layer takes 16 input channels (1 for each feature map)
    nn.ReLU(),
    nn.MaxPool2d(2), # image becomes 7x7
    nn.Flatten(), # We have 32 feature maps and 7x7=49 pixels, so next linear takes 32x49=1568 inputs
    nn.Linear(1568, 128), # Spew out 128 neurons
    nn.ReLU(),
    nn.Linear(128 , 10) # 10 neurons in the final layer, one for each digit
)

cnn_model = cnn_model.to(device) # Upload model to device


In [8]:
loss_ft = nn.CrossEntropyLoss() # We use CEL instead of BCELoss, since we have more than 2 classes.
opt = optim.Adam(cnn_model.parameters(), lr=0.01)

In [9]:
for epoch in range(10):
  # Total across all batches
  total_train_loss = 0
  total_accuracy = 0

  cnn_model.train()
  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device) # Use the GPU
    pred = cnn_model(X_batch) # Make predictions
    loss = loss_ft(pred, y_batch) # Calculate loss
    loss.backward() # Backpropogate
    opt.step() # Update weights
    opt.zero_grad() # Reset

    # Calculate accuracy of predictions
    predicted = pred.argmax(dim=1)
    correct = (predicted == y_batch).sum().item()

    # Calculate cross entropy loss
    total_train_loss += loss.item()
    total_accuracy += correct

  # Calculate the average across the batch
  avg_train_loss = total_train_loss / len(train_loader)
  acc = total_accuracy / len(train_dataset)

  # Test the model
  cnn_model.eval()
  total_test_correct = 0
  with torch.no_grad():
      for X_batch, y_batch in test_loader:
          X_batch, y_batch = X_batch.to(device), y_batch.to(device)
          pred = cnn_model(X_batch)
          total_test_correct += (pred.argmax(dim=1) == y_batch).sum().item()

  test_acc = total_test_correct / len(test_dataset)

  print(f'Epoch {epoch}')
  print(f'Training Cross-Entropy Loss: {avg_train_loss:.4f}')
  print(f'Training Accuracy: {acc:.4f}')

  print(f'\nTesting Accuracy: {test_acc:.4f}\n')

Epoch 0
Training Cross-Entropy Loss: 0.1391
Training Accuracy: 0.9569

Testing Accuracy: 0.9814

Epoch 1
Training Cross-Entropy Loss: 0.0625
Training Accuracy: 0.9808

Testing Accuracy: 0.9830

Epoch 2
Training Cross-Entropy Loss: 0.0533
Training Accuracy: 0.9842

Testing Accuracy: 0.9770

Epoch 3
Training Cross-Entropy Loss: 0.0484
Training Accuracy: 0.9853

Testing Accuracy: 0.9838

Epoch 4
Training Cross-Entropy Loss: 0.0465
Training Accuracy: 0.9864

Testing Accuracy: 0.9862

Epoch 5
Training Cross-Entropy Loss: 0.0448
Training Accuracy: 0.9870

Testing Accuracy: 0.9766

Epoch 6
Training Cross-Entropy Loss: 0.0435
Training Accuracy: 0.9883

Testing Accuracy: 0.9845

Epoch 7
Training Cross-Entropy Loss: 0.0419
Training Accuracy: 0.9888

Testing Accuracy: 0.9855

Epoch 8
Training Cross-Entropy Loss: 0.0426
Training Accuracy: 0.9887

Testing Accuracy: 0.9833

Epoch 9
Training Cross-Entropy Loss: 0.0433
Training Accuracy: 0.9887

Testing Accuracy: 0.9830

